In [ ]:
import pandas as pd

from experiments_2024.datasets import (
    load_weather,
    load_building,
)
from experiments_2024.zone_level_analysis import (
    base,
    cleaning,
    viz,
    regression_functions,
)

In [ ]:
pd.set_option("display.max_rows", 100000000)

# Helper constants and functions

In [ ]:
KWH_PER_TON_COOLING = 3.5168528
WH_PER_BTU = 0.293071
CFM_to_CMH = 0.0283168 / 60  # m3/hr
PROJECTS = ["OFF-2", "OFF-3", "OFF-4", "OFF-5", "OFF-6", "OFF-7"]
PROJECTS_TRIAL3 = ["OFF-2", "OFF-3", "OFF-4", "OFF-5", "OFF-6"]
SUMMER_START = pd.Timestamp("05-01-2024")
SUMMER_END = pd.Timestamp("10-01-2024")
ONLY_BUSINESS_HOURS = True
NO_WEEKENDS = True
TRIALS = ["All-Zone", "Trial 1", "Trial 2", "Trial 3"]

In [ ]:
def load_utility_data(
    projects,
    time_range,
    year="2024",
    field="C",
):
    utility = load_building(year, field).loc[time_range[0] : time_range[1], projects]
    if field == "C":
        utility = utility * KWH_PER_TON_COOLING  # to kWh
    if field == "H":
        utility = utility * WH_PER_BTU  # to kWh
    return utility

In [ ]:
SUMMARY_STATISTIC = "Mean"
MODE = "Absolute Change"

# OFF-2

In [ ]:
project = "OFF-2"
start = regression_functions.FORMAL_TRIALS_2024_START[project]
end = regression_functions.FORMAL_TRIALS_2024_END[project]

## Cooling

In [ ]:
T = cleaning.clean_df(
    load_weather("2024"),
    "dummy",
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
cooling = load_utility_data([project], (start, end + pd.Timedelta(days=1)), field="C")
cooling = cleaning.clean_by_column(
    df=cooling,
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
T, cooling = base.trim_to_common_elements(
    [T, cooling], clean_cols=False, clean_idx=True
)
T = T["temperature"]

In [ ]:
binary_df = regression_functions.get_2024_binary_df(
    project,
    freq="daily",
    baseline_column="Control",
    no_weekends=True,
    control_for_weekends=False,
    control_for_summer=True,
)

In [ ]:
reg_results = regression_functions.general_Delta_fn(
    df=cooling, T=T, binary=binary_df, mode=MODE, summary_statistic=SUMMARY_STATISTIC
)

In [ ]:
shapes = pd.Series(0, index=binary_df.index)
shapes.loc[shapes.index >= pd.Timestamp("08-19-2024")] = 1
shape_legend = {
    "series": shapes,
    "name": {0: "Summer", 1: "End Summer"},
    "shape": {0: "circle", 1: "x"},
}

In [ ]:
fig = viz.plot_experiment_regression(
    reg_results,
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
    line_legend={
        "color": {
            "Control": "Blue",
            "Trial 1": "Orange",
            "Trial 2": "Purple",
            "Trial 3": "Green",
            "All-Zone": "Red",
        },
    },
    shape_legend=shape_legend,
    y_axis_title=f"{SUMMARY_STATISTIC} Cooling (kWh)",
    additive_column_dict={"End Summer": ["Control", "Trial 3", "All-Zone"]},
    dont_add_to_legend=["Summer"],
    fig=None,
)

In [ ]:
fig

In [ ]:
reg_results.filter(like="P-Value").T

In [ ]:
reg_results.loc[
    :,
    ["Delta All-Zone", "Delta Trial 1", "Delta Trial 2", "Delta Trial 3"],
].T

In [ ]:
reg_results.loc[:, "R2"]

# OFF-3

In [ ]:
project = "OFF-3"
start = regression_functions.FORMAL_TRIALS_2024_START[project]
end = regression_functions.FORMAL_TRIALS_2024_END[project]

## Cooling

In [ ]:
T = cleaning.clean_df(
    load_weather("2024"),
    "dummy",
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
cooling = load_utility_data([project], (start, end + pd.Timedelta(days=1)), field="C")
cooling = cleaning.clean_by_column(
    df=cooling,
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
T, cooling = base.trim_to_common_elements(
    [T, cooling], clean_cols=False, clean_idx=True
)
T = T["temperature"]

In [ ]:
binary_df = regression_functions.get_2024_binary_df(
    project,
    freq="daily",
    baseline_column="Control",
    no_weekends=True,
    control_for_weekends=False,
    control_for_summer=True,
)

In [ ]:
reg_results = regression_functions.general_Delta_fn(
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
)

In [ ]:
shapes = pd.Series(0, index=binary_df.index)
shapes.loc[shapes.index >= pd.Timestamp("08-19-2024")] = 1
shape_legend = {
    "series": shapes,
    "name": {0: "Summer", 1: "End Summer"},
    "shape": {0: "circle", 1: "x"},
}

In [ ]:
fig = viz.plot_experiment_regression(
    reg_results,
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
    line_legend={
        "color": {
            "Control": "Blue",
            "Trial 1": "Orange",
            "Trial 2": "Purple",
            "Trial 3": "Green",
            "All-Zone": "Red",
        },
    },
    shape_legend=shape_legend,
    y_axis_title=f"{SUMMARY_STATISTIC} Cooling (kWh)",
    additive_column_dict={"End Summer": ["Control", "Trial 3", "All-Zone"]},
    dont_add_to_legend=["Summer"],
    fig=None,
)

In [ ]:
fig

In [ ]:
reg_results.filter(like="P-Value").T

In [ ]:
reg_results.loc[
    :,
    ["Delta All-Zone", "Delta Trial 1", "Delta Trial 2", "Delta Trial 3"],
].T

In [ ]:
reg_results.loc[:, "R2"]

# OFF-5

In [ ]:
project = "OFF-5"
start = regression_functions.FORMAL_TRIALS_2024_START[project]
end = regression_functions.FORMAL_TRIALS_2024_END[project]

## Cooling

In [ ]:
T = cleaning.clean_df(
    load_weather("2024"),
    "dummy",
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
cooling = load_utility_data([project], (start, end + pd.Timedelta(days=1)), field="C")
cooling = cleaning.clean_by_column(
    df=cooling,
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
T, cooling = base.trim_to_common_elements(
    [T, cooling], clean_cols=False, clean_idx=True
)
T = T["temperature"]

In [ ]:
binary_df = regression_functions.get_2024_binary_df(
    project,
    freq="daily",
    baseline_column="Control",
    no_weekends=True,
    control_for_weekends=False,
    # control_for_summer=True,
)

In [ ]:
reg_results = regression_functions.general_Delta_fn(
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
)

In [ ]:
shapes = pd.Series(0, index=binary_df.index)
shapes.loc[shapes.index >= pd.Timestamp("08-19-2024")] = 1
shape_legend = {
    "series": shapes,
    "name": {0: "Summer", 1: "End Summer"},
    "shape": {0: "circle", 1: "x"},
}

In [ ]:
fig = viz.plot_experiment_regression(
    reg_results,
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
    line_legend={
        "color": {
            "Control": "Blue",
            "Trial 1": "Orange",
            "Trial 2": "Purple",
            "Trial 3": "Green",
            "All-Zone": "Red",
        },
    },
    # shape_legend=shape_legend,
    y_axis_title=f"{SUMMARY_STATISTIC} Cooling (kWh)",
    # additive_column_dict={"End Summer": ["Control", "Trial 3", "All-Zone"]},
    dont_add_to_legend=["Summer"],
    fig=None,
)

In [ ]:
fig

In [ ]:
reg_results.filter(like="P-Value").T

In [ ]:
reg_results.loc[
    :,
    ["Delta All-Zone", "Delta Trial 1", "Delta Trial 2", "Delta Trial 3"],
].T

In [ ]:
reg_results.loc[:, "R2"]

# OFF-6

In [ ]:
project = "OFF-6"
start = regression_functions.FORMAL_TRIALS_2024_START[project]
end = regression_functions.FORMAL_TRIALS_2024_END[project]

## Cooling

In [ ]:
T = cleaning.clean_df(
    load_weather("2024"),
    "dummy",
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
cooling = load_utility_data([project], (start, end + pd.Timedelta(days=1)), field="C")
cooling = cleaning.clean_by_column(
    df=cooling,
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
T, cooling = base.trim_to_common_elements(
    [T, cooling], clean_cols=False, clean_idx=True
)
T = T["temperature"]

In [ ]:
binary_df = regression_functions.get_2024_binary_df(
    project,
    freq="daily",
    baseline_column="Control",
    no_weekends=True,
    control_for_weekends=False,
    # control_for_summer=True,
    delete_days=[pd.Timestamp("09-16-2024"), pd.Timestamp("09-17-2024")],
)

In [ ]:
reg_results = regression_functions.general_Delta_fn(
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
)

In [ ]:
shapes = pd.Series(0, index=binary_df.index)
shapes.loc[shapes.index >= pd.Timestamp("08-19-2024")] = 1
shape_legend = {
    "series": shapes,
    "name": {0: "Summer", 1: "End Summer"},
    "shape": {0: "circle", 1: "x"},
}

In [ ]:
fig = viz.plot_experiment_regression(
    reg_results,
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
    line_legend={
        "color": {
            "Control": "Blue",
            "Trial 1": "Orange",
            "Trial 2": "Purple",
            "Trial 3": "Green",
            "All-Zone": "Red",
        },
    },
    # shape_legend=shape_legend,
    y_axis_title=f"{SUMMARY_STATISTIC} Cooling (kWh)",
    # additive_column_dict={"End Summer": ["Control", "Trial 3", "All-Zone"]},
    dont_add_to_legend=["Summer"],
    fig=None,
)

In [ ]:
fig

In [ ]:
reg_results.filter(like="P-Value").T

In [ ]:
reg_results.loc[
    :,
    ["Delta All-Zone", "Delta Trial 1", "Delta Trial 2", "Delta Trial 3"],
].T

In [ ]:
reg_results.loc[:, "R2"]

# OFF-4

In [ ]:
project = "OFF-4"
start = regression_functions.FORMAL_TRIALS_2024_START[project]
end = regression_functions.FORMAL_TRIALS_2024_END[project]

## Cooling

In [ ]:
T = cleaning.clean_df(
    load_weather("2024"),
    "dummy",
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
cooling = load_utility_data([project], (start, end + pd.Timedelta(days=1)), field="C")
cooling = cleaning.clean_by_column(
    df=cooling,
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
T, cooling = base.trim_to_common_elements(
    [T, cooling], clean_cols=False, clean_idx=True
)
T = T["temperature"]

In [ ]:
binary_df = regression_functions.get_2024_binary_df(
    project,
    freq="daily",
    baseline_column="Control",
    no_weekends=True,
    control_for_weekends=False,
    control_for_summer=False,
    delete_days=[pd.Timestamp("09-16-2024"), pd.Timestamp("09-17-2024")],
    off4_all_zone="adjust",
)

In [ ]:
reg_results = regression_functions.general_Delta_fn(
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
)

In [ ]:
shapes = pd.Series(0, index=binary_df.index)
shapes.loc[shapes.index >= pd.Timestamp("08-19-2024")] = 1
shape_legend = {
    "series": shapes,
    "name": {0: "Summer", 1: "End Summer"},
    "shape": {0: "circle", 1: "x"},
}

In [ ]:
fig = viz.plot_experiment_regression(
    reg_results,
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
    line_legend={
        "color": {
            "Control": "Blue",
            "Trial 1": "Orange",
            "Trial 1 (Poor Control)": "Orange",
            "Trial 2": "Purple",
            "Trial 3": "Green",
            "All-Zone": "Red",
            "All-Zone (Poor Control)": "Red",
        },
        "style": {
            "Control": "solid",
            "Trial 1": "solid",
            "Trial 1 (Poor Control)": "dash",
            "Trial 2": "solid",
            "Trial 3": "solid",
            "All-Zone": "solid",
            "All-Zone (Poor Control)": "dash",
        },
        "opacity": {
            "Control": 1,
            "Trial 1": 1,
            "Trial 1 (Poor Control)": 0.5,
            "Trial 2": 1,
            "Trial 3": 1,
            "All-Zone": 1,
            "All-Zone (Poor Control)": 0.5,
        },
    },
    # shape_legend=shape_legend,
    y_axis_title=f"{SUMMARY_STATISTIC} Cooling (kWh)",
    # additive_column_dict={"End Summer": ["Control", "Trial 3", "All-Zone"]},
    dont_add_to_legend=["Summer"],
    fig=None,
    width=1000,
)

In [ ]:
fig

In [ ]:
reg_results.filter(like="P-Value").T

In [ ]:
reg_results.loc[
    :,
    ["Delta All-Zone", "Delta Trial 1", "Delta Trial 2", "Delta Trial 3"],
].T

In [ ]:
reg_results.loc[:, "R2"]

# OFF-7

In [ ]:
project = "OFF-7"
start = regression_functions.FORMAL_TRIALS_2024_START[project]
end = regression_functions.FORMAL_TRIALS_2024_END[project]

## Cooling

In [ ]:
T = cleaning.clean_df(
    load_weather("2024"),
    "dummy",
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
cooling = load_utility_data([project], (start, end + pd.Timedelta(days=1)), field="C")
cooling = cleaning.clean_by_column(
    df=cooling,
    only_business_hours=True,
    no_weekends=True,
    start_date=start,
    end_date=end + pd.Timedelta(days=1),
)

In [ ]:
T, cooling = base.trim_to_common_elements(
    [T, cooling], clean_cols=False, clean_idx=True
)
T = T["temperature"]

In [ ]:
binary_df = regression_functions.get_2024_binary_df(
    project,
    freq="daily",
    baseline_column="Control",
    no_weekends=True,
    control_for_weekends=False,
    control_for_summer=True,
    off7_trial_3="drop",
    # delete_days=[
    #    pd.Timestamp("09-16-2024"),
    #    pd.Timestamp("09-17-2024")
    # ]
)

In [ ]:
reg_results = regression_functions.general_Delta_fn(
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
)

In [ ]:
shapes = pd.Series(0, index=binary_df.index)
shapes.loc[shapes.index >= pd.Timestamp("08-19-2024")] = 1
shape_legend = {
    "series": shapes,
    "name": {0: "Summer", 1: "End Summer"},
    "shape": {0: "circle", 1: "x"},
}

In [ ]:
fig = viz.plot_experiment_regression(
    reg_results,
    df=cooling,
    T=T,
    binary=binary_df,
    mode=MODE,
    summary_statistic=SUMMARY_STATISTIC,
    line_legend={
        "color": {
            "Control": "Blue",
            "Trial 1": "Orange",
            "Trial 2": "Purple",
            "Trial 3": "Green",
            "Trial 3 (Bad)": "Green",
            "All-Zone": "Red",
        },
    },
    shape_legend=shape_legend,
    y_axis_title=f"{SUMMARY_STATISTIC} Cooling (kWh)",
    additive_column_dict={"End Summer": ["Control", "Trial 3", "All-Zone"]},
    dont_plot_lines=["Trial 3 (Bad)"],
    dont_plot_dots=["Trial 3 (Bad)"],
    dont_add_to_legend=["Trial 3 (Bad)", "Summer"],
    fig=None,
)

In [ ]:
fig

In [ ]:
reg_results.filter(like="P-Value").T

In [ ]:
reg_results.loc[
    :,
    [
        "Delta All-Zone",
        "Delta Trial 1",
        "Delta Trial 2",
    ],  # "Delta Trial 3"
].T

In [ ]:
reg_results.loc[:, "R2"]